# 01 — EDA (Phase 2)

Dataset: **Hillel Yaffe Glaucoma Dataset (HYGD)** v1.1.0, PhysioNet, DOI [10.13026/m92s-0z95](https://doi.org/10.13026/m92s-0z95). Open Data Commons Attribution License v1.0. 747 fundus images, 288 supplied patient IDs. Exact source hashes and counts were reverified on 2026-08-31. Dataset images and row-level records are intentionally excluded from the public repository.

In [ ]:
import sys
sys.path.insert(0, "..")
from src.data_utils import load_dataset_metadata

df = load_dataset_metadata("../data/raw")
print("Total images:", len(df))
print("Unique supplied patient IDs:", df["patient_id"].nunique())
{"rows": len(df), "negative_rows": int((df['label'] == 0).sum()), "positive_rows": int((df['label'] == 1).sum())}

## Class balance

73.4% GON+ (glaucomatous) vs 26.6% GON- — real imbalance, handled later via class-weighted loss, not oversampling.

In [ ]:
import matplotlib.pyplot as plt
counts = df["label"].value_counts().reindex([1, 0])
fig, ax = plt.subplots(figsize=(8, 6))
ax.bar(["GON+ (glaucomatous)", "GON- (non-glaucomatous)"], counts.values)
ax.set_ylabel("Number of images")
ax.set_title(f"HYGD class distribution ({len(df)} images, {df['patient_id'].nunique()} supplied patient IDs)")
fig.tight_layout()
fig.savefig("../figures/01_class_distribution.png", dpi=120)
plt.close(fig)
counts

![class distribution](../figures/01_class_distribution.png)

## Image quality scores (FundusQ-Net, 1-10)

In [ ]:
df["quality_score"].describe()

![quality score distribution](../figures/02_quality_score_distribution.png)

## Images per supplied patient ID

Mean 2.6 images per supplied patient ID, up to 14 for one ID. Supplied-ID grouping is necessary but not sufficient: the later integrity audit found exact duplicate files spanning different supplied IDs. The preferred evaluator therefore links duplicate-connected IDs before splitting.

In [ ]:
images_per_supplied_id = df.groupby("patient_id").size()
fig, ax = plt.subplots(figsize=(8, 6))
ax.hist(images_per_supplied_id, bins=range(1, int(images_per_supplied_id.max()) + 2))
ax.set_xlabel("Images per supplied patient ID")
ax.set_ylabel("Number of supplied patient IDs")
ax.set_title("Images-per-supplied-ID distribution")
fig.tight_layout()
fig.savefig("../figures/03_images_per_patient.png", dpi=120)
plt.close(fig)
images_per_supplied_id.describe()

![images per supplied patient ID](../figures/03_images_per_patient.png)

## Public-data boundary

No fundus example panel is committed. Users obtain HYGD directly from PhysioNet under its own data license and keep all image-level material local.